# Phase 6 — Décision, A/B Testing & Mesure d'Impact
## Dataset : Hillstrom E-Mail Analytics

**Objectif :** Formuler 3 recommandations actionnables, concevoir le plan 
A/B test et estimer l'impact financier.

### Plan de cette phase :
1. 3 Recommandations actionnables
2. Plan A/B Test complet
3. Métriques de suivi post-décision
4. Impact financier et opérationnel
5. Conclusion executive

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm

df = pd.read_csv('data/hillstrom.csv')
print("Donnees chargees :", df.shape)

Donnees chargees : (64000, 12)


## 1. Recommandations Actionnables

### Recommandation 1 — Cibler uniquement les Persuadables
**Priorité : HAUTE**

Au lieu d envoyer 42 614 emails a toute la base traitee,
concentrer la campagne sur les 15 999 Persuadables identifies
par le modele T-Learner (uplift score > P75).

Justification quantitative :
- Reduction du volume emails : -62%
- Budget economise : 13 308 $ par campagne
- Meme revenu incremental maintenu
- ROI campagne ameliore de +40%

### Recommandation 2 — Eviter absolument les Sleeping Dogs
**Priorité : HAUTE**

Ne jamais contacter les 7 026 clients identifies comme
Sleeping Dogs (uplift score < P25).

Justification quantitative :
- Ces clients convertissent a 1.68% SANS email
- L email reduit leur probabilite d achat
- Economie supplementaire : 3 513 $ par campagne
- Risque de desabonnement evite

### Recommandation 3 — Privilegier l email homme
**Priorité : MOYENNE**

Pour les Persuadables, privilegier la version email homme
plutot que email femme.

Justification quantitative :
- Uplift email homme : +0.68% vs baseline
- Uplift email femme : +0.31% vs baseline
- Email homme 2x plus efficace
- A/B test recommande pour confirmer sur prochaine campagne

## 1. Recommandations Actionnables

### Recommandation 1 — Cibler uniquement les Persuadables
**Priorité : HAUTE**

Au lieu d envoyer 42 614 emails a toute la base traitee,
concentrer la campagne sur les 15 999 Persuadables identifies
par le modele T-Learner (uplift score > P75).

Justification quantitative :
- Reduction du volume emails : -62%
- Budget economise : 13 308 $ par campagne
- Meme revenu incremental maintenu
- ROI campagne ameliore de +40%

### Recommandation 2 — Eviter absolument les Sleeping Dogs
**Priorité : HAUTE**

Ne jamais contacter les 7 026 clients identifies comme
Sleeping Dogs (uplift score < P25).

Justification quantitative :
- Ces clients convertissent a 1.68% SANS email
- L email reduit leur probabilite d achat
- Economie supplementaire : 3 513 $ par campagne
- Risque de desabonnement evite

### Recommandation 3 — Privilegier l email homme
**Priorité : MOYENNE**

Pour les Persuadables, privilegier la version email homme
plutot que email femme.

Justification quantitative :
- Uplift email homme : +0.68% vs baseline
- Uplift email femme : +0.31% vs baseline
- Email homme 2x plus efficace
- A/B test recommande pour confirmer sur prochaine campagne

In [2]:
print("=== CALCUL TAILLE ECHANTILLON ===\n")

# Parametres statistiques
alpha = 0.05        # Seuil de signification 5%
power = 0.80        # Puissance statistique 80%
p_baseline = 0.0057 # Taux de conversion baseline (No Email)
p_expected = 0.0125 # Taux de conversion attendu (Mens Email)
effect_size = p_expected - p_baseline

# Calcul taille echantillon
z_alpha = norm.ppf(1 - alpha/2)
z_beta = norm.ppf(power)

p_pooled = (p_baseline + p_expected) / 2
n = (2 * p_pooled * (1 - p_pooled) * (z_alpha + z_beta)**2) / effect_size**2

print(f"Parametres du test :")
print(f"  Alpha (seuil) : {alpha}")
print(f"  Puissance : {power}")
print(f"  Taux baseline : {p_baseline*100:.2f}%")
print(f"  Taux attendu : {p_expected*100:.2f}%")
print(f"  Effect size : {effect_size*100:.2f}%")
print(f"\nTaille echantillon minimale par groupe : {int(n):,}")
print(f"Taille totale minimale : {int(n*2):,}")
print(f"\nDuree estimee : 4 semaines")
print(f"Base client disponible : 64 000")
print(f"Faisabilite : {'OUI' if 64000 >= n*2 else 'NON'}")

=== CALCUL TAILLE ECHANTILLON ===

Parametres du test :
  Alpha (seuil) : 0.05
  Puissance : 0.8
  Taux baseline : 0.57%
  Taux attendu : 1.25%
  Effect size : 0.68%

Taille echantillon minimale par groupe : 3,061
Taille totale minimale : 6,122

Duree estimee : 4 semaines
Base client disponible : 64 000
Faisabilite : OUI


### Résultats du calcul statistique

| Paramètre | Valeur |
|---|---|
| Taille minimale par groupe | 3 061 clients |
| Taille totale minimale | 6 122 clients |
| Base disponible | 64 000 clients |
| Faisabilité | OUI — 10x au-dessus du minimum |
| Durée | 4 semaines |
| Puissance statistique | 80% |
| Seuil de signification | 5% |

### Métriques de suivi post-décision
- Taux de conversion par groupe (primaire)
- Revenu incremental par email envoye
- Taux de desabonnement
- ROI campagne
- Uplift score moyen des clients cibles

### Critère d arrêt
Arreter le test si p-value < 0.05 après 2 semaines
ou si taux de desabonnement > 2%

In [5]:
print("=== ANALYSE INCREMENTALE ===\n")

# Taux de conversion baseline (sans email)
baseline_rate = 0.0057

# Conversions incrementales uniquement
conv_incr_blast = int(clients_blast * (0.0125 - baseline_rate))
conv_incr_cible = int(clients_cibles * (0.0173 - baseline_rate))

revenu_incr_blast = conv_incr_blast * revenu_moyen_conversion
revenu_incr_cible = conv_incr_cible * revenu_moyen_conversion

profit_incr_blast = revenu_incr_blast - cout_blast
profit_incr_cible = revenu_incr_cible - cout_cible

roi_incr_blast = revenu_incr_blast / cout_blast * 100
roi_incr_cible = revenu_incr_cible / cout_cible * 100

print("ANALYSE SUR REVENU INCREMENTAL UNIQUEMENT :\n")
print(f"{'':30} {'Blast':>12} {'Ciblee':>12}")
print(f"{'-'*54}")
print(f"{'Emails envoyes':30} {clients_blast:>12,} {clients_cibles:>12,}")
print(f"{'Cout campagne ($)':30} {cout_blast:>12,.0f} {cout_cible:>12,.0f}")
print(f"{'Conv. incrementales':30} {conv_incr_blast:>12,} {conv_incr_cible:>12,}")
print(f"{'Revenu incremental ($)':30} {revenu_incr_blast:>12,.0f} {revenu_incr_cible:>12,.0f}")
print(f"{'Profit incremental ($)':30} {profit_incr_blast:>12,.0f} {profit_incr_cible:>12,.0f}")
print(f"{'ROI incremental (%)':30} {roi_incr_blast:>12,.0f} {roi_incr_cible:>12,.0f}")
print(f"\nEconomie budget : {cout_blast - cout_cible:,.0f} $")
print(f"Difference profit incremental : {profit_incr_cible - profit_incr_blast:,.0f} $")

=== ANALYSE INCREMENTALE ===

ANALYSE SUR REVENU INCREMENTAL UNIQUEMENT :

                                      Blast       Ciblee
------------------------------------------------------
Emails envoyes                       42,614       15,999
Cout campagne ($)                    21,307        8,000
Conv. incrementales                     289          185
Revenu incremental ($)               41,038       26,270
Profit incremental ($)               19,731       18,270
ROI incremental (%)                     193          328

Economie budget : 13,308 $
Difference profit incremental : -1,460 $


## 3. Conclusion Executive

### L arbitrage fondamental

| Métrique | Blast | Ciblée | Différence |
|---|---|---|---|
| Emails envoyés | 42 614 | 15 999 | -62% |
| Cout campagne | 21 307$ | 8 000$ | -13 308$ |
| Profit incremental | 19 731$ | 18 270$ | -1 460$ |
| ROI incremental | 193% | 328% | +70% |

### Recommandation finale

Pour économiser 13 308$ de budget campagne,
on accepte de perdre 1 460$ de profit incremental.

C est un ratio de 9:1 — pour chaque dollar de profit
sacrifié, on économise 9 dollars de budget.

Dans un contexte de contrainte budgétaire,
la campagne ciblée est clairement supérieure.

### Les 3 recommandations hiérarchisées
1. Cibler uniquement les 15 999 Persuadables
2. Exclure les 7 026 Sleeping Dogs
3. Privilégier l email homme (uplift 2x supérieur)